## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 128], stddev=0.1), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([128]), dtype=tf.float32)
        self.W2 = tf.Variable(tf.random.truncated_normal([128, 10], stddev=0.1), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])
        hidden = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(hidden, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [11]:
train_data, test_data = mnist_dataset()
for epoch in range(500):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.99865 ; accuracy 0.7797833
epoch 1 : loss 0.9964543 ; accuracy 0.7801
epoch 2 : loss 0.99427015 ; accuracy 0.78035
epoch 3 : loss 0.9920972 ; accuracy 0.78065
epoch 4 : loss 0.98993576 ; accuracy 0.78106666
epoch 5 : loss 0.9877857 ; accuracy 0.78148335
epoch 6 : loss 0.98564667 ; accuracy 0.78173333
epoch 7 : loss 0.9835189 ; accuracy 0.78213334
epoch 8 : loss 0.9814021 ; accuracy 0.78243333
epoch 9 : loss 0.9792965 ; accuracy 0.7827
epoch 10 : loss 0.97720176 ; accuracy 0.783
epoch 11 : loss 0.9751181 ; accuracy 0.7833833
epoch 12 : loss 0.9730452 ; accuracy 0.7837333
epoch 13 : loss 0.970983 ; accuracy 0.78426665
epoch 14 : loss 0.9689315 ; accuracy 0.78463334
epoch 15 : loss 0.96689063 ; accuracy 0.78495
epoch 16 : loss 0.96486056 ; accuracy 0.7852833
epoch 17 : loss 0.96284103 ; accuracy 0.7855
epoch 18 : loss 0.96083206 ; accuracy 0.78575
epoch 19 : loss 0.9588334 ; accuracy 0.78618336
epoch 20 : loss 0.9568451 ; accuracy 0.78643334
epoch 21 : loss 0.95486724 ; a